# ARYA RAG: Indexing Service (upload, ingest, retry, delete)

**Branch:** `shivam/dev/feature/rag-pgvector-indexing`
**Location:** `backend/app/rag/ingestion/`

This is the service that ties the earlier pieces together:

```text
bytes -> validate -> hash -> duplicate check -> store file -> create row
      -> extract + chunk -> embed -> replace_chunks (one transaction) -> status ready | failed
```

It depends only on **ports**: `RagRepository` (Person 2 implements it over PostgreSQL and pgvector), `Embedder`, a `FileStorage`, and the extraction functions. It never writes SQL.

## Contract changes this branch needs (update the contract notebook first)

| Change | Why |
|---|---|
| Add `storage_key: str` to `DocumentRecord` | Retry and delete must find the stored file. Keep it **internal**; the API must never return it. |
| Add `DocumentStateError(RagError)` | Raised when an action is invalid for the document's state (retrying a document that is not failed). |
| `create_document` must map a unique violation on `(owner_id, content_sha256)` to `IngestionError(DUPLICATE_DOCUMENT)` | Makes concurrent duplicate uploads safe. |

## Behaviour rules
1. Unsupported type, oversize and empty files are rejected **before** any row or file is created.
2. A duplicate of a `ready` or in-progress document raises `DUPLICATE_DOCUMENT`. A duplicate of a `failed` document reprocesses that same row, so no second row appears.
3. Failures after the row exists are recorded as `failed` with a safe code and message, and the result is returned (the UI shows it). They are not raised.
4. Unexpected errors are logged without document content and shown as a generic "Indexing failed."
5. `replace_chunks` runs in one transaction. If it fails, nothing is written and the document becomes `failed`.
6. Stored file names are generated internally. The user's filename is display-only and is stripped of paths and control characters.
7. Delete removes the database row first (chunks cascade), then the file. A leftover file is logged, never a leftover row.

In [ ]:
%pip install -q pypdf sentence-transformers

## 1. Imports and contract stand-ins

In [ ]:
from __future__ import annotations

import hashlib
import logging
import math
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, BinaryIO, Callable, Protocol
from uuid import UUID, uuid4

# In the repo these come from app.rag.contracts. The stand-ins below let the notebook
# run on its own; they mirror the contract's field names (plus the two additions
# listed at the top of this notebook).
try:
    from app.rag.contracts import (
        ChunkRecord, DocumentNotFoundError, DocumentRecord, DocumentStateError,
        DocumentStatus, FileType, IngestionError, IngestionErrorCode, IngestionResult,
        RagError, RetrievalError, RetrievalFilters, RetrievedChunk, SourceCitation)
except ImportError:
    class DocumentStatus(str, Enum):
        UPLOADED = "uploaded"; PROCESSING = "processing"; READY = "ready"; FAILED = "failed"

    class FileType(str, Enum):
        PDF = "pdf"; TXT = "txt"; MD = "md"

    class IngestionErrorCode(str, Enum):
        UNSUPPORTED_FILE_TYPE = "UNSUPPORTED_FILE_TYPE"
        FILE_TOO_LARGE = "FILE_TOO_LARGE"
        NO_EXTRACTABLE_TEXT = "NO_EXTRACTABLE_TEXT"
        EMBEDDING_FAILED = "EMBEDDING_FAILED"
        INDEXING_FAILED = "INDEXING_FAILED"
        DUPLICATE_DOCUMENT = "DUPLICATE_DOCUMENT"

    class RagError(Exception): pass

    class IngestionError(RagError):
        def __init__(self, code, message):
            super().__init__(message)
            self.code, self.message = code, message

    class RetrievalError(RagError): pass
    class DocumentNotFoundError(RagError): pass
    class DocumentStateError(RagError): pass          # NEW: invalid state transition

    def _utcnow(): return datetime.now(timezone.utc)

    @dataclass
    class DocumentRecord:
        id: UUID
        owner_id: str
        filename: str
        file_type: FileType
        content_sha256: str
        storage_key: str                              # NEW: internal, never returned by the API
        status: DocumentStatus
        chunk_count: int = 0
        error_code: IngestionErrorCode | None = None
        error_message: str | None = None
        created_at: datetime = field(default_factory=_utcnow)
        updated_at: datetime = field(default_factory=_utcnow)

    @dataclass
    class IngestionResult:
        document_id: UUID
        status: DocumentStatus
        chunk_count: int = 0
        error_code: IngestionErrorCode | None = None
        error_message: str | None = None

    @dataclass
    class ChunkRecord:
        chunk_index: int
        text: str
        token_count: int
        embedding: list
        page: int | None = None
        section: str | None = None

    @dataclass
    class RetrievedChunk:
        chunk_id: UUID
        document_id: UUID
        title: str
        text: str
        score: float
        page: int | None = None
        section: str | None = None

    @dataclass
    class SourceCitation:
        document_id: UUID
        title: str
        chunk_id: UUID
        excerpt: str
        page: int | None = None

        @classmethod
        def from_chunk(cls, chunk, max_chars: int = 300):
            return cls(chunk.document_id, chunk.title, chunk.chunk_id, chunk.text[:max_chars], chunk.page)

    @dataclass
    class RetrievalFilters:
        document_ids: list | None = None
        top_k: int = 5
        min_score: float | None = None

        def __post_init__(self):
            if not 1 <= self.top_k <= 20:
                raise ValueError("top_k must be between 1 and 20")

## 2. Test doubles

These mirror the contract so the notebook runs without a database or model. After conversion to modules, import them from `backend/tests/fakes/` instead.

In [ ]:
def _now() -> datetime:
    return datetime.now(timezone.utc)


def _cosine(a, b) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x * x for x in a)), math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0


class FakeEmbedder:
    """Hashed bag-of-words: deterministic, similar text scores higher."""
    def __init__(self, dimension: int = 64):
        self.dimension = dimension

    def _vec(self, text: str):
        v = [0.0] * self.dimension
        for w in re.findall(r"[a-z0-9]+", text.lower()):
            v[int(hashlib.sha256(w.encode()).hexdigest()[:8], 16) % self.dimension] += 1.0
        if not any(v):
            v[int(hashlib.sha256(text.encode()).hexdigest()[:8], 16) % self.dimension] = 1.0
        n = math.sqrt(sum(x * x for x in v))
        return [x / n for x in v]

    def embed_texts(self, texts): return [self._vec(t) for t in texts]
    def embed_query(self, text): return self._vec(text)


class InMemoryRagRepository:
    """Test double that follows the RagRepository contract (owner-scoped, ready-only search)."""
    def __init__(self):
        self.docs: dict = {}
        self.chunks: dict = {}
        self.fail_replace: Exception | None = None
        self.fail_search: Exception | None = None
        self.fail_create: Exception | None = None

    def create_document(self, owner_id, filename, file_type, content_sha256, storage_key):
        if self.fail_create:
            raise self.fail_create
        if self.find_by_hash(owner_id, content_sha256):
            raise IngestionError(IngestionErrorCode.DUPLICATE_DOCUMENT, "Document already uploaded.")
        rec = DocumentRecord(id=uuid4(), owner_id=owner_id, filename=filename, file_type=file_type,
                             content_sha256=content_sha256, storage_key=storage_key,
                             status=DocumentStatus.UPLOADED)
        self.docs[rec.id] = rec
        return rec

    def find_by_hash(self, owner_id, content_sha256):
        return next((d for d in self.docs.values()
                     if d.owner_id == owner_id and d.content_sha256 == content_sha256), None)

    def _owned(self, owner_id, document_id):
        d = self.docs.get(document_id)
        return d if d and d.owner_id == owner_id else None

    def set_status(self, owner_id, document_id, status, error_code=None, error_message=None):
        d = self._owned(owner_id, document_id)
        if not d:
            raise DocumentNotFoundError()
        d.status, d.error_code, d.error_message, d.updated_at = status, error_code, error_message, _now()

    def replace_chunks(self, owner_id, document_id, chunks):
        d = self._owned(owner_id, document_id)
        if not d:
            raise DocumentNotFoundError()
        if self.fail_replace:
            raise self.fail_replace                       # nothing written: mimics a rolled-back transaction
        self.chunks[document_id] = [(uuid4(), c) for c in chunks]
        d.status, d.chunk_count, d.updated_at = DocumentStatus.READY, len(chunks), _now()
        d.error_code = d.error_message = None

    def search_chunks(self, owner_id, query_embedding, filters):
        if self.fail_search:
            raise self.fail_search
        hits = []
        for d in self.docs.values():
            if d.owner_id != owner_id or d.status != DocumentStatus.READY:
                continue
            if filters.document_ids and d.id not in filters.document_ids:
                continue
            for cid, c in self.chunks.get(d.id, []):
                hits.append(RetrievedChunk(chunk_id=cid, document_id=d.id, title=d.filename, text=c.text,
                                           page=c.page, section=c.section,
                                           score=_cosine(query_embedding, c.embedding)))
        return sorted(hits, key=lambda h: h.score, reverse=True)[: filters.top_k]

    def list_documents(self, owner_id):
        return [d for d in self.docs.values() if d.owner_id == owner_id]

    def get_document(self, owner_id, document_id):
        return self._owned(owner_id, document_id)

    def delete_document(self, owner_id, document_id):
        if not self._owned(owner_id, document_id):
            return False
        del self.docs[document_id]
        self.chunks.pop(document_id, None)
        return True

## 3. File storage port

Original uploads go to **private** storage, never into PostgreSQL, Git or the frontend. Keys are generated here, so user input can never choose a path. `LocalFileStorage` also refuses any key that resolves outside its root.

In [ ]:
from pathlib import Path


class FileStorage(Protocol):
    def save(self, owner_id: str, data: bytes, extension: str) -> str: ...
    def read(self, key: str) -> bytes: ...
    def delete(self, key: str) -> None: ...


class LocalFileStorage:
    """Stores files under `root` as <owner-hash>/<uuid>.<ext>. Swap for S3 or similar later."""

    def __init__(self, root):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)

    def _path(self, key: str) -> Path:
        p = (self.root / key).resolve()
        if p != self.root and self.root not in p.parents:
            raise ValueError("Invalid storage key")
        return p

    def save(self, owner_id: str, data: bytes, extension: str) -> str:
        ext = re.sub(r"[^a-z0-9]", "", extension.lower())[:8] or "bin"
        owner_dir = hashlib.sha256(owner_id.encode()).hexdigest()[:16]
        key = f"{owner_dir}/{uuid4().hex}.{ext}"
        path = self._path(key)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_bytes(data)
        tmp.replace(path)                                  # atomic on the same filesystem
        return key

    def read(self, key: str) -> bytes:
        return self._path(key).read_bytes()

    def delete(self, key: str) -> None:
        self._path(key).unlink(missing_ok=True)


class InMemoryFileStorage:
    def __init__(self):
        self.files: dict[str, bytes] = {}

    def save(self, owner_id, data, extension):
        key = f"mem/{uuid4().hex}.{extension}"
        self.files[key] = data
        return key

    def read(self, key):
        return self.files[key]

    def delete(self, key):
        self.files.pop(key, None)

## 4. The service

The extraction functions are injected, so the service can be tested alone:
- `validate_fn(filename, data) -> FileType` (real: `detect_file_type`)
- `extract_fn(data, filename) -> (sha256, FileType, list[TextChunk])` (real: `extract_and_chunk`)

In [ ]:
log = logging.getLogger("arya.rag.ingestion")


def _safe_filename(name: str | None, limit: int = 255) -> str:
    """Display name only: no directories, no control characters."""
    name = (name or "").replace("\\", "/").rsplit("/", 1)[-1]
    name = re.sub(r"[\x00-\x1f\x7f]", "", name).strip()
    return name[:limit]


def _require_user(user_id: str) -> None:
    if not isinstance(user_id, str) or not user_id.strip():
        raise ValueError("user_id must be the verified owner ID")


class IngestionService:
    def __init__(self, repo, embedder, storage: FileStorage,
                 validate_fn: Callable[[str, bytes], FileType],
                 extract_fn: Callable[[bytes, str], tuple]):
        self._repo, self._embedder, self._storage = repo, embedder, storage
        self._validate, self._extract = validate_fn, extract_fn

    # ---- upload -------------------------------------------------------------
    def upload(self, user_id: str, filename: str, data: bytes) -> IngestionResult:
        _require_user(user_id)
        filename = _safe_filename(filename)
        file_type = self._validate(filename, data)            # raises; nothing created yet
        digest = hashlib.sha256(data).hexdigest()

        existing = self._repo.find_by_hash(user_id, digest)
        if existing is not None:
            if existing.status != DocumentStatus.FAILED:
                raise IngestionError(IngestionErrorCode.DUPLICATE_DOCUMENT,
                                     "This document was already uploaded.")
            return self._process(user_id, existing, data)     # retry the same row

        key = self._storage.save(user_id, data, file_type.value)
        try:
            doc = self._repo.create_document(user_id, filename, file_type, digest, key)
        except Exception:
            self._delete_file_quietly(key)                    # do not orphan the file
            raise
        return self._process(user_id, doc, data)

    # ---- RagService port: process an existing row ---------------------------
    def ingest_document(self, user_id: str, document_id: UUID, file: BinaryIO,
                        filename: str) -> IngestionResult:
        _require_user(user_id)
        doc = self._repo.get_document(user_id, document_id)
        if doc is None:
            raise DocumentNotFoundError()
        data = file.read()
        self._validate(_safe_filename(filename) or doc.filename, data)
        return self._process(user_id, doc, data)

    # ---- read ---------------------------------------------------------------
    def get_document(self, user_id: str, document_id: UUID) -> DocumentRecord:
        _require_user(user_id)
        doc = self._repo.get_document(user_id, document_id)
        if doc is None:                                       # also covers another user's document
            raise DocumentNotFoundError()
        return doc

    def list_documents(self, user_id: str) -> list[DocumentRecord]:
        _require_user(user_id)
        return self._repo.list_documents(user_id)

    # ---- retry / delete -----------------------------------------------------
    def retry(self, user_id: str, document_id: UUID) -> IngestionResult:
        doc = self.get_document(user_id, document_id)
        if doc.status != DocumentStatus.FAILED:
            raise DocumentStateError("Only failed documents can be retried.")
        try:
            data = self._storage.read(doc.storage_key)
        except Exception:
            log.warning("stored file missing for document %s", doc.id)
            return self._fail(user_id, doc.id, IngestionErrorCode.INDEXING_FAILED,
                              "The stored file could not be read. Please upload it again.")
        return self._process(user_id, doc, data)

    def delete(self, user_id: str, document_id: UUID) -> bool:
        _require_user(user_id)
        doc = self._repo.get_document(user_id, document_id)
        if doc is None:
            return False
        deleted = self._repo.delete_document(user_id, document_id)   # chunks cascade
        if deleted:
            self._delete_file_quietly(doc.storage_key)
        return deleted

    # ---- internals ----------------------------------------------------------
    def _process(self, user_id: str, doc: DocumentRecord, data: bytes) -> IngestionResult:
        self._repo.set_status(user_id, doc.id, DocumentStatus.PROCESSING)
        try:
            digest, _ftype, chunks = self._extract(data, doc.filename)
            if digest != doc.content_sha256:
                raise IngestionError(IngestionErrorCode.INDEXING_FAILED,
                                     "The file does not match the stored record.")
            if not chunks:
                raise IngestionError(IngestionErrorCode.NO_EXTRACTABLE_TEXT,
                                     "No extractable text found.")
            vectors = self._embedder.embed_texts([c.text for c in chunks])
            if len(vectors) != len(chunks):
                raise IngestionError(IngestionErrorCode.EMBEDDING_FAILED, "Embedding failed.")
            records = [ChunkRecord(chunk_index=c.chunk_index, text=c.text, token_count=c.token_count,
                                   page=c.page, section=c.section, embedding=v)
                       for c, v in zip(chunks, vectors)]
            self._repo.replace_chunks(user_id, doc.id, records)   # one transaction, sets READY
        except IngestionError as exc:
            return self._fail(user_id, doc.id, exc.code, exc.message)
        except Exception:
            log.exception("ingestion failed for document %s", doc.id)   # never log file content
            return self._fail(user_id, doc.id, IngestionErrorCode.INDEXING_FAILED, "Indexing failed.")
        return IngestionResult(document_id=doc.id, status=DocumentStatus.READY, chunk_count=len(records))

    def _fail(self, user_id, document_id, code, message) -> IngestionResult:
        self._repo.set_status(user_id, document_id, DocumentStatus.FAILED, code, message)
        return IngestionResult(document_id=document_id, status=DocumentStatus.FAILED,
                               error_code=code, error_message=message)

    def _delete_file_quietly(self, key: str) -> None:
        try:
            self._storage.delete(key)
        except Exception:
            log.warning("could not delete stored file %s", key)

## 5. Wiring in the real backend

```python
from app.rag.ingestion.config import ChunkingConfig
from app.rag.ingestion.extraction import detect_file_type
from app.rag.ingestion.pipeline import extract_and_chunk
from app.rag.embedding import SentenceTransformerEmbedder, assert_dimension

config = ChunkingConfig()
embedder = SentenceTransformerEmbedder()
assert_dimension(embedder, 384)                      # must match vector(384)

service = IngestionService(
    repo=PostgresRagRepository(session_factory),     # Person 2
    embedder=embedder,
    storage=LocalFileStorage("var/uploads"),         # private directory, outside the web root and Git
    validate_fn=lambda name, data: detect_file_type(name, data, config),
    extract_fn=lambda data, name: extract_and_chunk(data, name, config),
)
```

## 6. Tests

The extraction functions are replaced by small fakes here (the real ones live in the extraction notebook), so these tests check the **orchestration**: ordering, status transitions, duplicates, failure handling and cleanup.

In [ ]:
import tempfile

logging.getLogger("arya").addHandler(logging.NullHandler())   # keep expected-failure logs out of test output


@dataclass
class TextChunk:
    chunk_index: int
    text: str
    token_count: int
    page: int | None = None
    section: str | None = None


def fake_validate(filename: str, data: bytes) -> FileType:
    ext = filename.rsplit(".", 1)[-1].lower() if "." in filename else ""
    if ext not in {"pdf", "txt", "md"}:
        raise IngestionError(IngestionErrorCode.UNSUPPORTED_FILE_TYPE, "Unsupported file type.")
    if len(data) > 1000:
        raise IngestionError(IngestionErrorCode.FILE_TOO_LARGE, "File too large.")
    if not data:
        raise IngestionError(IngestionErrorCode.NO_EXTRACTABLE_TEXT, "The file is empty.")
    return FileType(ext)


def fake_extract(data: bytes, filename: str):
    paras = [p.strip() for p in data.decode("utf-8", "replace").split("\n\n") if p.strip()]
    if not paras:
        raise IngestionError(IngestionErrorCode.NO_EXTRACTABLE_TEXT, "No extractable text found.")
    ftype = FileType(filename.rsplit(".", 1)[-1].lower())
    chunks = [TextChunk(i, p, len(p.split()), page=1, section="Notes") for i, p in enumerate(paras)]
    return hashlib.sha256(data).hexdigest(), ftype, chunks


def make_service():
    repo, storage = InMemoryRagRepository(), InMemoryFileStorage()
    svc = IngestionService(repo, FakeEmbedder(64), storage, fake_validate, fake_extract)
    return svc, repo, storage


def expect(exc, fn):
    try:
        fn()
    except exc as e:
        return e
    raise AssertionError(f"expected {exc.__name__}")


ALICE, BOB = "user_alice", "user_bob"
NOTES = b"Support vectors define the margin.\n\nKernels handle nonlinear data."

# --- happy path: ready, chunks stored with metadata, file stored
svc, repo, storage = make_service()
res = svc.upload(ALICE, "svm.md", NOTES)
assert res.status == DocumentStatus.READY and res.chunk_count == 2
doc = svc.get_document(ALICE, res.document_id)
assert doc.status == DocumentStatus.READY and doc.chunk_count == 2
assert len(storage.files) == 1 and storage.read(doc.storage_key) == NOTES
stored = repo.chunks[doc.id]
assert [c.chunk_index for _, c in stored] == [0, 1]
assert all(len(c.embedding) == 64 and c.page == 1 and c.section == "Notes" for _, c in stored)

# --- the uploaded document is searchable by its owner only
q = FakeEmbedder(64).embed_query("what are support vectors")
assert repo.search_chunks(ALICE, q, RetrievalFilters())[0].text.startswith("Support")
assert repo.search_chunks(BOB, q, RetrievalFilters()) == []

# --- duplicates: same user rejected, other user allowed, only one row kept
e = expect(IngestionError, lambda: svc.upload(ALICE, "copy.md", NOTES))
assert e.code == IngestionErrorCode.DUPLICATE_DOCUMENT
assert len(svc.list_documents(ALICE)) == 1 and len(storage.files) == 1
assert svc.upload(BOB, "svm.md", NOTES).status == DocumentStatus.READY

# --- early rejections create no row and no file
svc, repo, storage = make_service()
assert expect(IngestionError, lambda: svc.upload(ALICE, "a.docx", b"x")).code == IngestionErrorCode.UNSUPPORTED_FILE_TYPE
assert expect(IngestionError, lambda: svc.upload(ALICE, "a.txt", b"x" * 2000)).code == IngestionErrorCode.FILE_TOO_LARGE
assert expect(IngestionError, lambda: svc.upload(ALICE, "a.txt", b"")).code == IngestionErrorCode.NO_EXTRACTABLE_TEXT
assert repo.docs == {} and storage.files == {}

# --- extraction failure is recorded, returned and not raised
res = svc.upload(ALICE, "blank.txt", b"   \n\n  ")
assert res.status == DocumentStatus.FAILED and res.error_code == IngestionErrorCode.NO_EXTRACTABLE_TEXT
rec = svc.get_document(ALICE, res.document_id)
assert rec.status == DocumentStatus.FAILED and rec.error_code == IngestionErrorCode.NO_EXTRACTABLE_TEXT
assert repo.chunks.get(rec.id) is None

# --- embedding failure
class FailingEmbedder(FakeEmbedder):
    def embed_texts(self, texts):
        raise IngestionError(IngestionErrorCode.EMBEDDING_FAILED, "Embedding failed.")

svc, repo, storage = make_service()
svc._embedder = FailingEmbedder(64)
res = svc.upload(ALICE, "svm.md", NOTES)
assert res.status == DocumentStatus.FAILED and res.error_code == IngestionErrorCode.EMBEDDING_FAILED

# --- unexpected error: generic message, no leak, status failed, nothing written
svc, repo, storage = make_service()
repo.fail_replace = RuntimeError("boom secret /var/db/credentials")
res = svc.upload(ALICE, "svm.md", NOTES)
assert res.status == DocumentStatus.FAILED and res.error_code == IngestionErrorCode.INDEXING_FAILED
assert "secret" not in res.error_message and res.error_message == "Indexing failed."
assert repo.chunks == {}                                   # no partial chunks

# --- re-upload of a FAILED document reuses the same row (no duplicate)
repo.fail_replace = None
failed_id = res.document_id
res2 = svc.upload(ALICE, "svm.md", NOTES)
assert res2.document_id == failed_id and res2.status == DocumentStatus.READY
assert len(svc.list_documents(ALICE)) == 1

# --- retry reads the stored file
svc, repo, storage = make_service()
repo.fail_replace = RuntimeError("transient")
res = svc.upload(ALICE, "svm.md", NOTES)
assert res.status == DocumentStatus.FAILED
repo.fail_replace = None
retried = svc.retry(ALICE, res.document_id)
assert retried.status == DocumentStatus.READY and retried.chunk_count == 2
expect(DocumentStateError, lambda: svc.retry(ALICE, res.document_id))        # now ready
expect(DocumentNotFoundError, lambda: svc.retry(BOB, res.document_id))       # not Bob's

# --- retry when the stored file is gone
svc, repo, storage = make_service()
repo.fail_replace = RuntimeError("x")
res = svc.upload(ALICE, "svm.md", NOTES)
storage.files.clear()
out = svc.retry(ALICE, res.document_id)
assert out.status == DocumentStatus.FAILED and "upload it again" in out.error_message

# --- delete removes row, chunks and file; other users cannot delete
svc, repo, storage = make_service()
res = svc.upload(ALICE, "svm.md", NOTES)
assert svc.delete(BOB, res.document_id) is False and len(storage.files) == 1
assert svc.delete(ALICE, res.document_id) is True
assert repo.docs == {} and repo.chunks == {} and storage.files == {}
assert svc.delete(ALICE, res.document_id) is False

# --- a failed create_document does not orphan the stored file
svc, repo, storage = make_service()
repo.fail_create = RuntimeError("db down")
expect(RuntimeError, lambda: svc.upload(ALICE, "svm.md", NOTES))
assert storage.files == {}

# --- unknown or foreign documents look the same
expect(DocumentNotFoundError, lambda: svc.get_document(ALICE, uuid4()))

# --- ingest_document (RagService port) processes an existing row, and rejects mismatched bytes
import io
svc, repo, storage = make_service()
row = repo.create_document(ALICE, "svm.md", FileType.MD, hashlib.sha256(NOTES).hexdigest(), "k")
assert svc.ingest_document(ALICE, row.id, io.BytesIO(NOTES), "svm.md").status == DocumentStatus.READY
row2 = repo.create_document(ALICE, "other.md", FileType.MD, "0" * 64, "k2")
bad = svc.ingest_document(ALICE, row2.id, io.BytesIO(NOTES), "other.md")
assert bad.status == DocumentStatus.FAILED and "does not match" in bad.error_message
expect(DocumentNotFoundError, lambda: svc.ingest_document(BOB, row.id, io.BytesIO(NOTES), "svm.md"))

# --- filename handling is display-only and safe
assert _safe_filename("../../etc/passwd.md") == "passwd.md"
assert _safe_filename("C:\\Users\\me\\notes.txt") == "notes.txt"
assert _safe_filename("bad\x00name\n.md") == "badname.md"
assert len(_safe_filename("a" * 400 + ".md")) == 255
svc, repo, storage = make_service()
res = svc.upload(ALICE, "../../secret.md", NOTES)
assert svc.get_document(ALICE, res.document_id).filename == "secret.md"
assert "secret" not in next(iter(storage.files))           # storage key never uses the filename

# --- missing owner
expect(ValueError, lambda: svc.upload("", "a.md", NOTES))

# --- LocalFileStorage: round trip, private keys, traversal blocked
with tempfile.TemporaryDirectory() as tmp:
    ls = LocalFileStorage(Path(tmp) / "uploads")
    key = ls.save(ALICE, b"hello", "pdf")
    assert ls.read(key) == b"hello" and key.endswith(".pdf") and ALICE not in key
    expect(ValueError, lambda: ls.read("../outside.txt"))
    expect(ValueError, lambda: ls.delete("../../etc/passwd"))
    ls.delete(key)
    assert not (ls.root / key).exists()
    ls.delete(key)                                          # deleting twice is harmless

print("All indexing service tests passed")

## 7. Move this into the repository

| Notebook cells | Target file |
|---|---|
| 1 contract part | already in `app/rag/contracts.py` after you apply the three contract changes |
| 3 (storage) | `backend/app/rag/ingestion/storage.py` |
| 4 (service) | `backend/app/rag/ingestion/service.py` |
| 2 (test doubles) | `backend/tests/fakes/` |
| 6 (tests) | `backend/tests/unit/rag/test_ingestion_service.py`, rewritten as `pytest` functions |

**Before the PR**
1. Apply the three contract changes and get Person 2's approval on `storage_key` and the duplicate mapping.
2. Wait for Person 2's repository. Then add one **integration test** against a real PostgreSQL with pgvector: upload, ready, search, delete.
3. Run **Restart and Run All**, then **Clear All Outputs**, then commit.

**Known limits**
- A process crash during processing leaves the document in `processing`. Add a timeout that marks stale rows `failed` before relying on retry for this case.
- Synchronous processing blocks the request. Move to a background worker when files get large.
- Deleting the file after the row means a failed file delete leaves an orphan file (logged), never an orphan row.
- The same file under two different names by the same user is treated as a duplicate.